# [실습②] AI 응답 품질 평가 — DeepEval

2일차 3교시. 서비스의 답변을 **관련성 · 근거 충실도 · 정확성(G-Eval)**으로 채점하고, 채점 사유를 읽습니다.

In [ ]:
# 실습 저장소 내려받기 + API 키 등록
# API 키는 왼쪽 🔑(보안 비밀) 메뉴에 OPENAI_API_KEY 이름으로 등록해 둡니다. 코드에 직접 붙여 넣지 않습니다.
![ -d /content/ai-quality-lab ] || git clone -q https://github.com/leejaehee1/ai-quality-lab.git /content/ai-quality-lab

import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["PROMPTFOO_DISABLE_TELEMETRY"] = "1"
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

In [ ]:
!pip install -q deepeval pandas

In [ ]:
# 공통 예제 서비스 실행 (같은 런타임 안에서 http://localhost:8000 으로 호출)
# 팀 서비스를 테스트하는 조는 이 셀을 건너뛰고, 아래 SERVICE_URL만 바꿉니다.
!pip install -q fastapi uvicorn openai
!cd /content/ai-quality-lab/sample-service && nohup uvicorn main:app --port 8000 > server.log 2>&1 &

import time, requests
for _ in range(30):
    try:
        print(requests.get("http://localhost:8000/health", timeout=2).json())
        break
    except Exception:
        time.sleep(1)
else:
    print("서버가 뜨지 않았습니다. server.log를 확인하세요:")
    print(open("/content/ai-quality-lab/sample-service/server.log").read())

In [ ]:
import requests

SERVICE_URL = "http://localhost:8000/chat"   # 팀 서비스면 주소를 바꿉니다

def ask(question: str) -> dict:
    """서비스에 질문하고 답변·근거·토큰 사용량을 돌려준다. 팀 API 형식에 맞게 고칩니다."""
    r = requests.post(SERVICE_URL, json={"message": question}, timeout=60)
    r.raise_for_status()
    data = r.json()
    return {"answer": data["answer"],
            "contexts": data.get("sources", []),
            "usage": data.get("usage")}

print(ask("얼마씩 몇 달 줘?")["answer"])

## 골든셋으로 테스트 케이스 만들기

1일차 템플릿 `goldenset.csv`에서 정상 질문(자격·경계값·금액·신청)만 사용합니다. 우리 팀 골든셋으로 바꿔도 됩니다.

In [ ]:
import pandas as pd
from deepeval.test_case import LLMTestCase

golden = pd.read_csv("/content/ai-quality-lab/day1/templates/goldenset.csv")
golden = golden[golden["type"].isin(["자격", "경계값", "금액", "신청"])]

cases = []
for _, g in golden.iterrows():
    out = ask(g["question"])
    cases.append(LLMTestCase(
        input=g["question"],
        actual_output=out["answer"],
        expected_output=g["expected"],
        retrieval_context=out["contexts"] or None,
    ))
print(len(cases), "개 테스트 케이스")

## 지표 정의와 평가 실행

근거 문서(`sources`)를 돌려주지 않는 서비스는 충실도(Faithfulness)를 계산할 수 없어 자동으로 뺍니다.

In [ ]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, GEval
from deepeval.test_case import LLMTestCaseParams

JUDGE = "gpt-4.1-mini"

relevancy = AnswerRelevancyMetric(threshold=0.7, model=JUDGE)
faithfulness = FaithfulnessMetric(threshold=0.8, model=JUDGE)
correctness = GEval(
    name="정확성",
    criteria=(
        "실제 답변(actual output)이 기대 답변(expected output)의 핵심 사실"
        "(나이, 금액, 기간, 신청 방법)을 빠짐없이 정확하게 담고 있는지 평가한다. "
        "기대 답변과 모순되는 사실이 있으면 크게 감점한다."
    ),
    evaluation_params=[LLMTestCaseParams.INPUT,
                       LLMTestCaseParams.ACTUAL_OUTPUT,
                       LLMTestCaseParams.EXPECTED_OUTPUT],
    threshold=0.7,
    model=JUDGE,
)

metrics = [relevancy, correctness]
if all(c.retrieval_context for c in cases):
    metrics.append(faithfulness)

result = evaluate(test_cases=cases, metrics=metrics)

## 결과 표로 정리하기

In [ ]:
rows = []
for tr in result.test_results:
    for m in tr.metrics_data:
        rows.append({"질문": tr.input, "지표": m.name, "점수": round(m.score or 0, 2),
                     "통과": m.success, "사유": m.reason})
scores = pd.DataFrame(rows)
scores.pivot_table(index="질문", columns="지표", values="점수")

In [ ]:
# 채점 사유 읽기 — 점수만 보면 무엇을 고칠지 알 수 없습니다
pd.set_option("display.max_colwidth", None)
scores[~scores["통과"]][["질문", "지표", "점수", "사유"]]

### 해석 가이드

| 증상 | 의심할 곳 |
| --- | --- |
| 관련성 낮음 | 프롬프트가 질문 의도를 놓침 |
| 충실도 낮음 (환각) | 근거 문서에 없는 내용을 지어냄 |
| 정확성 낮음 + 충실도 높음 | 검색이 엉뚱한 문서를 가져옴 |
| 실행마다 점수가 크게 흔들림 | 서비스나 심판의 비결정성 → 반복 측정 |

**과제**: 실패한 항목 1개를 골라 사람이 직접 채점해 보고, 심판(LLM)의 판단과 비교합니다.